# 🔌 Setup SSH para Control Remoto de Colab

**Instrucciones:**
1. Asegúrate de seleccionar un **Runtime con GPU** (Runtime → Change runtime type → T4 GPU)
2. Ejecuta la **Celda 1** para instalar y configurar SSH
3. Ejecuta la **Celda 2** para obtener el comando de conexión
4. **Copia el hostname** que aparece y compártelo con el asistente

¡Eso es todo! Después de esto, el asistente tomará el control.

In [ ]:
# Celda 1: Instalar cloudflared y configurar SSH
import subprocess
import os

# Instalar cloudflared
print("📥 Instalando cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null

# Configurar SSH server
print("🔐 Configurando servidor SSH...")
password = "colab2025"
!echo "root:{password}" | chpasswd
!apt-get install -y openssh-server > /dev/null 2>&1
!echo 'PermitRootLogin yes' >> /etc/ssh/sshd_config
!echo 'PasswordAuthentication yes' >> /etc/ssh/sshd_config
!service ssh start

print("")
print("✅ SSH configurado correctamente.")
print(f"🔑 Contraseña SSH: {password}")
print("")
print("Ahora ejecuta la Celda 2 para iniciar el túnel.")

In [ ]:
# Celda 2: Iniciar túnel cloudflared  
import subprocess
import time
import re

print("🚀 Iniciando túnel cloudflared...")
print("(Esto puede tardar unos segundos)")
print("")

# Iniciar cloudflared en background
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Esperar a que se establezca el túnel y capturar la URL
time.sleep(10)

# Leer stderr donde cloudflared imprime la URL
import select
hostname = None
for _ in range(30):
    line = proc.stderr.readline().decode()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://([\w-]+\.trycloudflare\.com)', line)
        if match:
            hostname = match.group(1)
            break
    time.sleep(1)

if hostname:
    print("="*60)
    print("✅ ¡TÚNEL ESTABLECIDO!")
    print("="*60)
    print("")
    print(f"🌐 Hostname: {hostname}")
    print(f"🔑 Password: colab2025")
    print("")
    print("📋 Copia y comparte el hostname con el asistente.")
    print("="*60)
    print("")
    print("⚠️ MANTÉN ESTA CELDA EJECUTÁNDOSE. No cierres esta pestaña.")
    
    # Mantener vivo
    try:
        proc.wait()
    except:
        pass
else:
    print("❌ No se pudo obtener el hostname automáticamente.")
    print("Revisa la salida de cloudflared manualmente:")
    print(proc.stderr.read().decode()[:2000])